Generalized Additive Models:

Contributions:
1. https://web.stanford.edu/class/stats202/notes/Nonlinear/GAM.html
2. https://academic.oup.com/jrsssc/article/64/1/139/7067572#397191485
3. https://medium.com/just-another-data-scientist/building-interpretable-models-with-generalized-additive-models-in-python-c4404eaf5515
4. https://ecogambler.netlify.app/blog/autocorrelated-gams/
5. https://www.geeksforgeeks.org/artificial-intelligence/generalized-additive-model-in-python/

In [ ]:
# Import libraries:
import json
import pickle
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

from itertools import combinations
from pygam import LogisticGAM, s
from sklearn.model_selection import BaseCrossValidator, ParameterGrid
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, f1_score, recall_score, precision_score,
    roc_auc_score, average_precision_score, log_loss,
    roc_curve, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)

In [ ]:
# ============================================
# Paths & factor pairs
# ============================================
# Alternate machine path (SARB laptop):
# BASE_DIR = Path('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/legacy')
BASE_DIR = Path('C:/Users/ccwis/OneDrive/Documents/1. MEng - Data Science/Project/factor_timing/legacy')

DATA_DIR = BASE_DIR / 'data_input'
OUTPUT_DIR = BASE_DIR / 'results' / 'model_outputs'
PROBS_DIR = OUTPUT_DIR / 'probs'
ARTIFACT_DIR = OUTPUT_DIR / 'artifacts'
SHAP_DIR = OUTPUT_DIR / 'shap'
METRICS_PATH = OUTPUT_DIR / 'metrics_summary.csv'

for _dir in (PROBS_DIR, ARTIFACT_DIR, SHAP_DIR):
    _dir.mkdir(parents=True, exist_ok=True)

# Every factor pair this pipeline runs for a given model.
FACTOR_PAIRS = ['Momentum_vs_Value', 'Momentum_vs_Quality', 'Value_vs_Quality']

In [ ]:
# ============================================
# Data loading (parameterized by factor pair)
# ============================================
def load_data(pair):
    df_factor = pd.read_csv(DATA_DIR / 'labeled_relative_factor_returns.csv', parse_dates=['Date'])
    df_factor = df_factor[['Date', f'{pair}_trend']]
    df_factor = df_factor.set_index('Date')

    df = pd.read_csv(DATA_DIR / 'lagged_data.csv', parse_dates=['Date'])
    df = df.set_index('Date')

    data = pd.concat([df, df_factor], axis=1, join='inner')
    # Trend scanning needs returns out to t+10, so the final ~10 dates have
    # no computable label. ffill() would manufacture them from the last known
    # label, putting fabricated targets inside the OOS block -- drop instead.
    data = data.dropna(subset=[f'{pair}_trend'])
    # Unified missing-value policy across all three models.
    return data.ffill().bfill()

In [ ]:
# ============================================
# GAM Formula Builder
# ============================================
def build_gam_formula(feature_variables):
    """Build smooth spline terms for all features."""
    gam_formula = s(0)
    for i in range(1, len(feature_variables)):
        gam_formula += s(i)
    return gam_formula

In [ ]:
# ============================================
# Combinatorial Purged CV Splitter
# ============================================
CV_SPLITS = 5
CV_TEST_SPLITS = 2       # C(5,2) = 10 folds. k=1 collapsed CPCV to plain
                         # 5-fold purged CV with one backtest path.
PURGE_GAP = 12
FINAL_TEST_POINTS = 156  # ~3 years of unseen data

class CombinatorialPurgedCV(BaseCrossValidator):
    def __init__(self, n_splits=CV_SPLITS, n_test_splits=CV_TEST_SPLITS, purge_gap=PURGE_GAP):
        self.n_splits = n_splits
        self.n_test_splits = n_test_splits
        self.purge_gap = purge_gap

    def split(self, X, y=None, groups=None):
        n_samples = len(X)
        fold_size = n_samples // self.n_splits
        indices = np.arange(n_samples)
        fold_bounds = [(i * fold_size, (i + 1) * fold_size) for i in range(self.n_splits)]
        fold_bounds[-1] = (fold_bounds[-1][0], n_samples)

        for test_folds in combinations(range(self.n_splits), self.n_test_splits):
            test_mask = np.zeros(n_samples, dtype=bool)
            for f in test_folds:
                s_, e_ = fold_bounds[f]
                test_mask[s_:e_] = True

            train_mask = ~test_mask
            for f in test_folds:
                s_, e_ = fold_bounds[f]
                train_mask[max(0, s_ - self.purge_gap): s_] = False
                train_mask[e_: min(n_samples, e_ + self.purge_gap)] = False

            yield indices[train_mask], indices[test_mask]

    def get_n_splits(self, X=None, y=None, groups=None):
        from math import comb
        return comb(self.n_splits, self.n_test_splits)

    def group_bounds(self, n_samples):
        """(start, end) of each chronological group; the last takes the remainder."""
        size = n_samples // self.n_splits
        bounds = [(i * size, (i + 1) * size) for i in range(self.n_splits)]
        bounds[-1] = (bounds[-1][0], n_samples)
        return bounds

    def backtest_paths(self):
        """For each back-test path, the split whose held-out predictions supply each group.

        Every group is held out in C(N-1, k-1) splits; path j takes the j-th of them in
        split order, giving phi[N, k] = C(N-1, k-1) paths, each covering the whole sample
        out of sample (Lopez de Prado, 2018, ch. 12).
        """
        splits = list(combinations(range(self.n_splits), self.n_test_splits))
        holding = [[sp for sp, test in enumerate(splits) if g in test] for g in range(self.n_splits)]
        return [[holding[g][j] for g in range(self.n_splits)] for j in range(len(holding[0]))]


def held_out_vector(n_samples, test_idx, probs):
    """Full-length vector with a split's held-out probabilities in place, NaN elsewhere."""
    out = np.full(n_samples, np.nan)
    out[test_idx] = probs
    return out


def path_aucs(cv, y, split_probs):
    """ROC-AUC of every back-test path, assembled from the splits' held-out predictions."""
    bounds = cv.group_bounds(len(y))
    aucs = []
    for path in cv.backtest_paths():
        pred = np.full(len(y), np.nan)
        for group, split_no in enumerate(path):
            start, end = bounds[group]
            pred[start:end] = split_probs[split_no][start:end]
        assert not np.isnan(pred).any(), "a back-test path is missing held-out predictions"
        aucs.append(roc_auc_score(y, pred))
    return aucs

In [ ]:
def window_signature(X):
    """
    Create a stable, compact signature for a rolling window
    based on its mean and std across features.
    """
    import hashlib
    stats = np.concatenate([X.mean(axis=0), X.std(axis=0)])
    stats_bytes = stats.tobytes()
    return hashlib.sha1(stats_bytes).hexdigest()

In [ ]:
def tune_hyperparameters(X, y, feature_variables, return_grid=False, verbose=True):
    cv = CombinatorialPurgedCV()

    # lam=100 was selected at the grid ceiling on 2 of 3 pairs with CV AUC
    # still rising, and n_splines=5 at the grid floor on the same 2, so both
    # ends are extended. max_iter is a convergence setting rather than a
    # hyperparameter and is fixed. 20 configs vs 36.
    #
    # n_splines must be > spline_order (3), so 4 is the lowest valid value.
    # A rejected n_splines also leaves a shared TermList pinned at the bad
    # value, which then fails every later config -- hence a fresh formula per
    # fit below rather than one shared across the grid.
    param_grid = {
        "lam": [1, 10, 100, 1000, 10000],
        "n_splines": [4, 5, 7, 10],
        "max_iter": [1000],
        "fit_intercept": [True],
    }

    best_score, best_params = -np.inf, None
    rows = []

    grid = list(ParameterGrid(param_grid))
    if verbose:
        print(f"Testing {len(grid)} parameter combinations...")

    for params in grid:
        fold_scores, split_probs, failure = [], [], None

        for split_no, (train_idx, test_idx) in enumerate(cv.split(X, y)):

            # A setting must be scored on every split. One that cannot be fitted
            # on a split is disqualified rather than averaged over fewer splits,
            # which would compare settings on different subsets of the data.
            if len(np.unique(y[test_idx])) < 2 or len(np.unique(y[train_idx])) < 2:
                failure = f"split {split_no}: single-class labels"
                break

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")

                    gam = LogisticGAM(
                        build_gam_formula(feature_variables),
                        **params
                    ).fit(X[train_idx], y[train_idx])

                y_prob = gam.predict_proba(X[test_idx])

            except Exception as exc:
                failure = f"split {split_no}: {type(exc).__name__}"
                break

            fold_scores.append(float(roc_auc_score(y[test_idx], y_prob)))
            split_probs.append(held_out_vector(len(y), test_idx, y_prob))

        if failure:
            rows.append({**params, "mean_auc": np.nan, "failed": failure})
            if verbose:
                print(f"Disqualified {params}: {failure}")
            continue

        # Selection score: mean ROC-AUC over the ten splits. The back-test
        # paths pool predictions from models fitted on different groups, and
        # holding a group out moves the training base rate against that
        # group's own, so an intercept-only model scores about 0.44 on the
        # paths. Path AUCs are recorded for reporting, not used for selection.
        mean_score = float(np.mean(fold_scores))
        p_aucs = path_aucs(cv, y, split_probs)

        rows.append({
            **params,
            "mean_auc": mean_score,          # mean split AUC (selection score)
            "std_auc": np.std(fold_scores, ddof=1),
            "min_auc": np.min(fold_scores),
            "max_auc": np.max(fold_scores),
            "fold_aucs": fold_scores,
            "mean_path_auc": float(np.mean(p_aucs)),
            "path_aucs": p_aucs,             # back-test path AUCs (reporting)
            "failed": None
        })

        if mean_score > best_score:
            best_score, best_params = mean_score, params
            if verbose:
                print(f"New best params: {best_params} (ROC AUC: {best_score:.3f})")

    if best_params is None:
        raise RuntimeError("Every parameter setting failed on at least one CPCV split.")

    grid_df = pd.DataFrame(rows).sort_values("mean_auc", ascending=False)

    if verbose:
        print("\n=== Top 5 Parameter Combinations ===")
        for _, r in grid_df.head(5).iterrows():
            row_params = {k: r[k] for k in param_grid}
            print(f"Score: {r['mean_auc']:.3f} | Params: {row_params}")

    if return_grid:
        return best_params, grid_df
    return best_params

In [ ]:
def save_artifact(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    if isinstance(obj, pd.DataFrame):
        obj.to_csv(path, index=False)
    elif path.suffix == ".json":
        with open(path, "w") as f:
            json.dump(obj, f, indent=2)
    else:
        with open(path, "wb") as f:
            pickle.dump(obj, f)

In [ ]:
# ============================================
# Walk-Forward Validation
# ============================================
def walk_forward_validation(
    X,
    y,
    feature_variables,
    n_test,
    base_params,
    lookback=None,  # None = expanding window
    retune_every=52,
    selections=None  # list that records each retuned parameter set
):
    """
    GAM walk-forward validation,
    minimal configuration:
      - lookback window size (None = expanding)
      - retune frequency
    """

    actuals, preds, probs = [], [], []

    current_params = base_params.copy()

    hyperparam_cache = {}

    print(f"\nRunning GAM walk-forward validation...")
    print(f"Training window = {lookback if lookback else 'expanding (all history)'}")
    print(f"Retune every = {retune_every}")
    print(f"Total test points = {n_test}")

    for i in tqdm(range(n_test)):

        train_size = len(X) - n_test + i
        if train_size <= 0:
            continue

        # ===== Purge gap =====
        # Trend-scanning labels (span=(1,12)) resolve using returns out to
        # t+10, so training rows within PURGE_GAP of the test point have
        # labels computed from the test period itself. Drop them.
        train_end = train_size - PURGE_GAP
        if train_end <= 0:
            continue

        # Expanding window: train on all history up to the purged end.
        # (lookback=None). Set lookback to an int for a fixed rolling window.
        start = 0 if lookback is None else max(0, train_end - lookback)
        X_train = X[start:train_end]
        y_train = y[start:train_end]

        X_test  = X[train_size:train_size+1]
        y_test  = y[train_size:train_size+1]

        # ===== scaling =====
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test  = scaler.transform(X_test)

        # ===== retuning =====
        if retune_every and i > 0 and i % retune_every == 0:

            sig = window_signature(X_train)

            if sig in hyperparam_cache:
                current_params = hyperparam_cache[sig]
            else:
                current_params = tune_hyperparameters(X_train, y_train, feature_variables)
                hyperparam_cache[sig] = current_params
            if selections is not None:
                selections.append({"from_step": i, "n_train": len(y_train), **current_params})

        # ===== model training =====
        # Fresh formula per fit: LogisticGAM mutates the TermList's n_splines,
        # so a shared formula would carry state across retuned parameter sets.
        gam = LogisticGAM(
            build_gam_formula(feature_variables), **current_params
        ).fit(X_train, y_train)

        # ===== predict =====
        y_prob = gam.predict_proba(X_test)
        y_pred = (y_prob >= 0.5).astype(int)

        actuals.append(y_test[0])
        preds.append(y_pred[0])
        probs.append(y_prob[0])

    print("\n=== Final Evaluation Results ===")
    print(classification_report(actuals, preds))

    return (
        np.array(actuals),
        np.array(preds),
        np.array(probs)
    )

In [ ]:
# ============================================
# Results Visualization
# ============================================
def display_walkforward_results(y_true, y_pred, y_proba, pair_label=""):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)

    auc = roc_auc_score(y_true, y_proba)
    pr_auc = average_precision_score(y_true, y_proba)

    y_proba_clipped = np.clip(y_proba, 1e-12, 1 - 1e-12)
    ll = log_loss(y_true, y_proba_clipped)

    print(f"Walk-forward Performance [{pair_label}]:")
    print(f"Accuracy : {acc:.3f}")
    print(f"F1 Score : {f1:.3f}")
    print(f"Recall   : {rec:.3f}")
    print(f"Precision: {prec:.3f}")
    print(f"ROC AUC  : {auc:.3f}")
    print(f"PR  AUC  : {pr_auc:.3f}")
    print(f"Log Loss : {ll:.3f}")

    plt.figure(figsize=(12,5))
    plt.plot(y_true, label="True", marker="o",linestyle='None')
    plt.plot(y_pred, label="Predicted", marker="x",linestyle='None')
    plt.plot(y_proba,label="Probability",linestyle='--', alpha=0.5)
    plt.title(f"Walk-forward Predictions vs True Values [{pair_label}]")
    plt.legend()
    plt.show()

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    plt.figure(figsize=(6,6))
    plt.plot(fpr, tpr, label=f"ROC (AUC={auc:.3f})")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.plot([0,1],[0,1], '--', color='gray')
    plt.title(f"ROC Curve [{pair_label}]")
    plt.legend()
    plt.show()

    precs, recs, _ = precision_recall_curve(y_true, y_proba)
    plt.figure(figsize=(6,6))
    plt.plot(recs, precs, label=f"PR (AP={pr_auc:.3f})", color="purple")
    plt.title(f"Precision-Recall Curve [{pair_label}]")
    plt.legend()
    plt.show()

    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(cm).plot(cmap="Blues")
    plt.title(f"Confusion Matrix [{pair_label}]")
    plt.show()

    return {
        "accuracy": acc, "f1": f1, "recall": rec, "precision": prec,
        "roc_auc": auc, "pr_auc": pr_auc, "log_loss": ll,
    }

In [ ]:
# ============================================
# Metrics summary — single source of truth for the write-up.
# Re-running a pair updates its row instead of duplicating it.
# ============================================
def append_metrics_row(row):
    row = {"timestamp": datetime.now(timezone.utc).isoformat(), **row}
    new_row = pd.DataFrame([row])
    if METRICS_PATH.exists():
        existing = pd.read_csv(METRICS_PATH)
        existing = existing[~(
            (existing["model"] == row["model"]) & (existing["factor_pair"] == row["factor_pair"])
        )]
        combined = pd.concat([existing, new_row], ignore_index=True)
    else:
        combined = new_row
    combined.to_csv(METRICS_PATH, index=False)

In [ ]:
# ============================================
# GAM SHAP (log-odds output, training distribution as background)
# ============================================
def compute_shap_importance(X, y, best_params, feature_variables, n_test):
    import shap

    train_end = len(X) - n_test - PURGE_GAP   # the purged initial tuning sample
    X_train = X[:train_end]
    y_train = y[:train_end]

    # Standardise to match the scale the walk-forward models were fit on;
    # previously this fitted on raw features while the walk-forward used scaled.
    X_train = StandardScaler().fit_transform(X_train)

    final_gam = LogisticGAM(
        build_gam_formula(feature_variables),
        **best_params
    ).fit(X_train, y_train)

    def gam_predict_log_odds(X_in):
        p = final_gam.predict_proba(X_in)
        p = np.asarray(p).astype(float)
        # Clip for numerical stability then logit
        eps = 1e-6
        p = np.clip(p, eps, 1 - eps)
        return np.log(p / (1 - p))  # log-odds

    # X_train as background keeps the baseline consistent with the training distribution
    explainer = shap.Explainer(gam_predict_log_odds, X_train)
    shap_values = explainer(X_train)

    importance = pd.Series(
        np.abs(shap_values.values).mean(axis=0),
        index=feature_variables
    ).sort_values(ascending=False)

    return importance, shap_values, X_train

In [ ]:
# ============================================
# Full pipeline for one factor pair
# ============================================
def run_pipeline(pair):
    pair_slug = pair.lower()
    print(f"\n{'='*60}\nFACTOR PAIR: {pair}\n{'='*60}")

    data = load_data(pair)

    X = data.iloc[:, :-1].values
    y = data.iloc[:, -1].values
    feature_variables = data.columns[:-1].tolist()
    X = np.nan_to_num(X, nan=0.0, posinf=1e10, neginf=-1e10)

    print("=== PHASE 1: HYPERPARAMETER TUNING ===")
    # Tune on pre-test data only -- tuning on the full sample selects
    # hyperparameters using the OOS labels.
    # Purge the PURGE_GAP weeks before the evaluation block: their trend-scanning labels
    # draw on its first weeks (the retuning searches use purged windows too).
    X_train_full = X[:-(FINAL_TEST_POINTS + PURGE_GAP)]
    y_train_full = y[:-(FINAL_TEST_POINTS + PURGE_GAP)]
    # Standardise with a scaler fit on pre-test data only, so tuning sees the
    # same feature scale that walk_forward_validation fits on.
    X_train_full_scaled = StandardScaler().fit_transform(X_train_full)
    best_params, tuning_grid = tune_hyperparameters(
        X_train_full_scaled, y_train_full, feature_variables, return_grid=True)
    save_artifact(tuning_grid, ARTIFACT_DIR / f"gam_{pair_slug}_tuning_grid.pkl")
    save_artifact(best_params, ARTIFACT_DIR / f"gam_{pair_slug}_best_params.json")

    print("\n=== PHASE 2: FINAL EVALUATION ===")
    selections = [{"from_step": 0, "n_train": len(y_train_full), **best_params}]
    actuals, preds, probs = walk_forward_validation(
        X,
        y,
        feature_variables,
        FINAL_TEST_POINTS,
        best_params,
        lookback=None,  # expanding window
        retune_every=52,
        selections=selections
    )
    save_artifact(selections, ARTIFACT_DIR / f"gam_{pair_slug}_walkforward_params.json")

    print("\n=== ADAPTIVE PARAMS RESULTS ===")
    metrics = display_walkforward_results(actuals, preds, probs, pair_label=pair)

    df_probs = pd.DataFrame({
        "Date": data.index[-len(probs):],
        "actual": actuals,
        "prediction": preds,
        "probability": probs,
    }).set_index("Date")
    df_probs.to_csv(PROBS_DIR / f"gam_{pair_slug}_probs.csv", index=True)

    shap_importance, shap_values, X_train = compute_shap_importance(
        X, y, best_params, feature_variables, FINAL_TEST_POINTS
    )
    shap_importance.head(20).to_csv(SHAP_DIR / f"gam_{pair_slug}_shap.csv", index=True)

    append_metrics_row({
        "model": "GAM",
        "factor_pair": pair,
        "n_test": FINAL_TEST_POINTS,
        **metrics,
        "best_params": json.dumps(best_params),
    })

    return {
        "pair": pair,
        "best_params": best_params,
        "tuning_grid": tuning_grid,
        "feature_names": feature_variables,
        "adaptive": (actuals, preds, probs),
        "shap_importance": shap_importance,
        "shap_values": shap_values,
        "X_train": X_train,
    }

In [ ]:
def plot_pair_diagnostics(result):
    import shap

    print(f"SHAP summary [{result['pair']}]")
    shap.summary_plot(result["shap_values"], result["X_train"],
                      feature_names=result["feature_names"], plot_type="dot")

In [ ]:
# ============================================
# Run every factor pair
# ============================================
all_results = {}
for pair in FACTOR_PAIRS:
    all_results[pair] = run_pipeline(pair)
    plot_pair_diagnostics(all_results[pair])

print("\nDone. Metrics summary written to:", METRICS_PATH)